In [1]:
import sys
from pathlib import Path

# Setup root progetto
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pre_tagger import PreTagger
from src.config import (
    TAG_ASSIGN_THRESHOLD,
    TAG_WEIGHT_COSINE,
    TAG_WEIGHT_OVERLAP,
)

print(">> Moduli caricati")

>> Moduli caricati


In [2]:
# --- Tag candidati da assegnare durante il pre-tagging ---
# Modifica questa lista con i tag che vuoi far valutare da TagAssigner
"""
candidate_tags = [
    "tag_1",
    "tag_2",
    "tag_3",
]
"""

# Tags per il DataSet 1
candidate_tags = [
    "Vector Retrieval & Embeddings",
    "Knowledge Graph & Entities",
    "Evaluation & Self-Correction",
    "Query Expansion & HyDE",
    "Hierarchical Summarization",
    "Fine-Tuning & Model Comparison"
]

# Path del file sidecar su cui verranno salvati i tag assegnati
sidecar_path = project_root / "data/processed/test/sidecar_ds1multibase_04_05T.json"

# Dimensione della pagina di scroll da Qdrant (controlla anche ogni quanti
# chunk viene fatta una scrittura sul sidecar, vedi PreTagger.run)
BATCH_SIZE = 100

print(f"?> Tag candidati: {candidate_tags}")
print(f"?> Sidecar: {sidecar_path}")

?> Tag candidati: ['Vector Retrieval & Embeddings', 'Knowledge Graph & Entities', 'Evaluation & Self-Correction', 'Query Expansion & HyDE', 'Hierarchical Summarization', 'Fine-Tuning & Model Comparison']
?> Sidecar: /home/jovyan/tesi_graphrag/data/processed/test/sidecar_ds1multibase_04_05T.json


In [3]:
print(f"  >>TagAssignThreshold: {TAG_ASSIGN_THRESHOLD}")
print(f"  >>TagWeightCosine: {TAG_WEIGHT_COSINE}")
print(f"  >>TagWeightOverlap: {TAG_WEIGHT_OVERLAP}")

with PreTagger(sidecar_path=sidecar_path) as pretagger:
    pretagger.run(
        candidate_tags=candidate_tags,
        batch_size=BATCH_SIZE,
    )

  >>TagAssignThreshold: 0.32
  >>TagWeightCosine: 0.75
  >>TagWeightOverlap: 0.25


/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


<<! File sidecar ripristinato allo stato iniziale !>>


Pre-tagging 'ds1_multilingual_base': 100%|██████████| 985/985 [01:40<00:00,  9.81chunk/s] 


In [4]:
from src.sidecar_manager import SidecarManager

# Istanza di sola lettura del sidecar per ispezionare il risultato:
#  poiché chiusa quella usata dal pre-tagger, non darà errori
#  NB: per riutilizzare questo notebook spegni tutti i Kernel che utilizzano lo stesso sidecar.json
#      se così non fosse ci sarebbero 2 sidecar_manager distinti e la problematica delle 2 diverse cache.
#      Tra diversi notebook però non c'è nessun metodo di controllo o avviso.
verifier = SidecarManager(filepath=str(sidecar_path))
global_tags = verifier.get_global_tags()

# Prova globale
print(f"!>> Tag globali registrati: {len(global_tags)}")
for tag, info in global_tags.items():
    print(f"  - {tag}: conteggio={info.get('count')}, colore={info.get('color')}")

# Campione dei chunk taggati
sidecar_data = verifier.load_data()
overrides = sidecar_data.get("tag_overrides", {})
print(f"\n!>> Totale chunk con tag assegnati: {len(overrides)}")

if overrides:
    print("Sample primi 5 chunk:")
    for chunk_id, info in list(overrides.items())[:5]:
        print(f"  - [{chunk_id}] -> {info.get('user_tags')}")

verifier.release_path()

!>> Tag globali registrati: 6
  - Fine-Tuning & Model Comparison: conteggio=369, colore=#4e79a7
  - Evaluation & Self-Correction: conteggio=188, colore=#f28e2b
  - Vector Retrieval & Embeddings: conteggio=182, colore=#e15759
  - Hierarchical Summarization: conteggio=51, colore=#76b7b2
  - Query Expansion & HyDE: conteggio=53, colore=#59a14f
  - Knowledge Graph & Entities: conteggio=79, colore=#edc949

!>> Totale chunk con tag assegnati: 551
Sample primi 5 chunk:
  - [Original_RAG_Lewis_chunk_73] -> ['Fine-Tuning & Model Comparison']
  - [RAPTOR_Hierarchical_RAG_chunk_61] -> ['Evaluation & Self-Correction']
  - [Self_RAG_chunk_39] -> ['Fine-Tuning & Model Comparison']
  - [Original_RAG_Lewis_chunk_12] -> ['Vector Retrieval & Embeddings']
  - [HyDE_Hypothetical_Embeddings_chunk_3] -> ['Vector Retrieval & Embeddings', 'Evaluation & Self-Correction', 'Fine-Tuning & Model Comparison']
